In [9]:
import pandas as pd
import re
from jiwer import wer
import ast
import Levenshtein
import numpy as np
import ast

# normalization library
import unicodedata
import contractions
from num2words import num2words

# google cloud library
from googleapiclient import discovery
from google.auth import default

In [10]:
# For Excel files, use read_excel instead
all_datasets = pd.read_excel("results/all_result_processed_normalized.xlsx", index_col=False)

In [11]:
def analyze_medical_text(project_id, location, text_content):
    """
    Call Google Healthcare API to analyze medical entities in text.
    Returns full API response payload for downstream mention reconstruction.
    """
    try:
        credentials, _ = default()
        service = discovery.build("healthcare", "v1", credentials=credentials)

        nlp_service_name = f"projects/{project_id}/locations/{location}/services/nlp"
        body = {"documentContent": text_content}

        response = service.projects().locations().services().nlp().analyzeEntities(
            nlpService=nlp_service_name,
            body=body,
        ).execute()

        return response
    except Exception as e:
        print(f"Error calling healthcare API: {e}")
        return {}



def _collect_mentions_from_response(response, mention_types=None):
    """
    Extract mention spans in a normalized shape:
    {begin, end, replacement, type}
    
    Args:
        response: API response dict
        mention_types: List of mention types to include (e.g., ['ANATOMICAL_STRUCTURE', 'BF_RESULT'])
                      If None, all mention types are included.
    """
    if not response:
        return []

    entities = response.get("entities", []) or []
    entity_mentions = response.get("entityMentions", []) or []

    mentions_to_replace = []

    # Preferred path: top-level entityMentions usually contains reliable offsets.
    if entity_mentions:
        for mention in entity_mentions:
            # Filter by mention type if specified
            mention_type = mention.get("type")
            if mention_types and mention_type not in mention_types:
                continue

            text_obj = mention.get("text", {}) or {}
            surface_text = text_obj.get("content", "")
            begin_offset = text_obj.get("beginOffset")

            if begin_offset is None or not surface_text:
                continue

            end_offset = begin_offset + len(surface_text)
            # Format: [MENTION_TYPE: TextSpan]
            replacement_tag = f"[{mention_type}: {surface_text}]"

            mentions_to_replace.append(
                {
                    "begin": begin_offset,
                    "end": end_offset,
                    "replacement": replacement_tag,
                    "type": mention_type,
                }
            )

        return mentions_to_replace

    # Fallback path: some responses only include entity->mentions.
    for entity in entities:
        mentions = entity.get("mentions", []) or []
        for mention in mentions:
            mention_type = mention.get("type")
            if mention_types and mention_type not in mention_types:
                continue

            text_obj = mention.get("text", {}) or {}
            surface_text = text_obj.get("content", "")
            begin_offset = text_obj.get("beginOffset")

            if begin_offset is None or not surface_text:
                continue

            end_offset = begin_offset + len(surface_text)
            # Format: [MENTION_TYPE: TextSpan]
            replacement_tag = f"[{mention_type}: {surface_text}]"
            mentions_to_replace.append(
                {
                    "begin": begin_offset,
                    "end": end_offset,
                    "replacement": replacement_tag,
                    "type": mention_type,
                }
            )

    return mentions_to_replace



def reconstruct_text_with_ner_tags(text_content, response, mention_types=None):
    """
    Reconstruct the original text with inline NER tags.
    Replaces mention spans with format: [MENTION_TYPE: TextSpan]
    
    Args:
        text_content: Original text
        response: API response dict
        mention_types: List of mention types to include (e.g., ['ANATOMICAL_STRUCTURE', 'BF_RESULT'])
                      If None, all mention types are included.
    """
    if pd.isna(text_content) or text_content is None:
        return text_content

    text_content = str(text_content)
    mentions_to_replace = _collect_mentions_from_response(response, mention_types=mention_types)
    if not mentions_to_replace:
        return text_content

    # Sort by reverse offset to avoid shifting indices during replacement.
    mentions_to_replace.sort(key=lambda x: x["begin"], reverse=True)

    # Skip overlapping spans to prevent malformed replacements.
    reconstructed_text = text_content
    last_begin = len(text_content) + 1
    for mention in mentions_to_replace:
        begin = mention["begin"]
        end = mention["end"]
        replacement = mention["replacement"]

        if begin < 0 or end > len(reconstructed_text) or begin >= end:
            continue

        if end > last_begin:
            continue

        reconstructed_text = (
            reconstructed_text[:begin] + replacement + reconstructed_text[end:]
        )
        last_begin = begin

    return reconstructed_text



def apply_ner_to_row(text_content, project_id, location, mention_types=None):
    """
    Apply NER analysis to one transcript row and return the tagged transcript.
    
    Args:
        text_content: Text to analyze
        project_id: GCP project ID
        location: GCP location
        mention_types: List of mention types to include. If None, all types included.
                      Example: ['ANATOMICAL_STRUCTURE', 'BF_RESULT', 'LAB_VALUE']
    """
    if pd.isna(text_content) or text_content is None:
        return text_content

    text_content = str(text_content)
    if not text_content.strip():
        return text_content

    response = analyze_medical_text(project_id, location, text_content)
    return reconstruct_text_with_ner_tags(text_content, response, mention_types=mention_types)

In [12]:
# Apply NER processing to create norm_human_transcript_ner column
PROJECT_ID = "bio-ramp-ner"
LOCATION = "us-central1"

print(f"Processing {len(all_datasets)} rows for NER tagging with mention types...")
print("Tag format: [MENTION_TYPE: TextSpan]\n")

all_datasets['norm_human_transcript_ner'] = all_datasets['norm_human_transcript'].apply(
    lambda text: apply_ner_to_row(text, PROJECT_ID, LOCATION, mention_types=None)
)

print("✓ NER tagging with mention types complete!")
print(f"Created column 'norm_human_transcript_ner' with {len(all_datasets)} rows")
print("\n" + "=" * 80)
print("Sample output (with mention type tags):")
print("=" * 80)

for idx in range(min(3, len(all_datasets))):
    print(f"\nRow {idx}:")
    tagged = all_datasets['norm_human_transcript_ner'].iloc[idx]
    print(f"Tagged: {str(tagged)[:250]}...")

Processing 120 rows for NER tagging with mention types...
Tag format: [MENTION_TYPE: TextSpan]

✓ NER tagging with mention types complete!
Created column 'norm_human_transcript_ner' with 120 rows

Sample output (with mention type tags):

Row 0:
Tagged: good morning i am doctor smith from [MEDICINE: babylon] can you just confirm your name date of birth and the first line of your address please hi my name is [MEDICINE: susan]  thirty redbridge street sw two two hz hello and your date of birth forty o...

Row 1:
Tagged:  hello hi i am doctor jacob and welcome to babylon hi  hi so just before we start is it alright if you could confirm your name for me please yep  john doe okay and your date of birth  uhh  twentyone twelve and nineteen  eightysix and your address for...

Row 2:
Tagged: hi there good morning hello good morning  i am doctor [MEDICINE: deen mirza] from gp at hand nice to see you nice to see you okay before we start your appointment could you please tell me your first name and

In [13]:
# Apply NER to each ASR transcript so we can detect medical entities on the hypothesis side too.
# These feed the union (human-OR-ASR) gating used in the medical-error reconstruction.
print("Running NER on ASR transcripts (whisper, phi4, parakeet)...")

all_datasets['norm_whisper_asr_ner'] = all_datasets['norm_whisper_asr'].apply(
    lambda t: apply_ner_to_row(t, PROJECT_ID, LOCATION, mention_types=None)
)
all_datasets['norm_phi4_asr_ner'] = all_datasets['norm_phi4_asr'].apply(
    lambda t: apply_ner_to_row(t, PROJECT_ID, LOCATION, mention_types=None)
)
all_datasets['norm_parakeet_asr_ner'] = all_datasets['norm_parakeet_asr'].apply(
    lambda t: apply_ner_to_row(t, PROJECT_ID, LOCATION, mention_types=None)
)

print("✓ ASR NER tagging complete: norm_whisper_asr_ner, norm_phi4_asr_ner, norm_parakeet_asr_ner")

Running NER on ASR transcripts (whisper, phi4, parakeet)...
✓ ASR NER tagging complete: norm_whisper_asr_ner, norm_phi4_asr_ner, norm_parakeet_asr_ner


In [20]:
def align_words(ref, hyp, ref_range):
    if pd.isna(ref) or pd.isna(hyp):
        return None
    
    ref = ref.split()
    hyp = hyp.split()
    lexicon = list(set(ref + hyp))
    word2digit = {word: chr(i) for i, word in enumerate(lexicon)}
    ref_uni = [word2digit[w] for w in ref]
    hyp_uni = [word2digit[w] for w in hyp]
    
    edit_ops = pd.DataFrame(Levenshtein.editops(''.join(ref_uni), ''.join(hyp_uni)), columns=['operation', 'ref_ix', 'hyp_ix'])
    aligned_ref, aligned_hyp = ref.copy(), hyp.copy()
    aligned_ops = ['='] * len(ref)
    aligned_ref_ix, aligned_hyp_ix = list(range(len(ref))), list(range(len(hyp)))
    ix_edit_ops = [np.nan] * len(aligned_ref)

    ins_count, del_count = 0, 0
    for idx, ops in edit_ops.iterrows():
        if ops['operation'] == 'insert':
            aligned_ref.insert(ins_count + ops['ref_ix'], '_')
            aligned_ops.insert(ins_count + ops['ref_ix'], 'ins')
            aligned_ref_ix.insert(ins_count + ops['ref_ix'], None)
            ix_edit_ops.insert(ins_count + ops['ref_ix'], idx)
            ins_count += 1
        elif ops['operation'] == 'delete':
            aligned_hyp.insert(del_count + ops['hyp_ix'], '_')
            aligned_ops[ins_count + ops['ref_ix']] = 'del'
            aligned_hyp_ix.insert(del_count + ops['hyp_ix'], None)
            ix_edit_ops[ins_count + ops['ref_ix']] = idx
            del_count += 1
        elif ops['operation'] == 'replace':
            aligned_ops[ins_count + ops['ref_ix']] = 'sub'
            ix_edit_ops[ins_count + ops['ref_ix']] = idx

    aligned_df = pd.DataFrame({
        'ref_ix': aligned_ref_ix,
        'hyp_ix': aligned_hyp_ix,
        'reference': aligned_ref,
        'hypothesis': aligned_hyp,
        'operation': aligned_ops,
        'index_edit_ops': ix_edit_ops
    }).astype({'ref_ix': 'Int32', 'hyp_ix': 'Int32', 'index_edit_ops': 'Int32'})


    return aligned_df
# apply align_words to whisper_norm_asr and phi4_norm_asr
all_datasets['whisper_aligned_df'] = all_datasets.apply(
    lambda row: align_words(row['norm_human_transcript'], row['norm_whisper_asr'], None), axis=1
)
all_datasets['phi4_aligned_df'] = all_datasets.apply(
    lambda row: align_words(row['norm_human_transcript'], row['norm_phi4_asr'], None), axis=1
)
all_datasets['parakeet_aligned_df'] = all_datasets.apply(
    lambda row: align_words(row['norm_human_transcript'], row['norm_parakeet_asr'], None), axis=1
)
# all_datasets['granite_aligned_df'] = all_datasets.apply(
#     lambda row: align_words(row['norm_human_transcript'], row['norm_granite'], None), axis=1
# )

In [21]:
# use output of aligned words to analyze equal, deletions, insertions, substitutions and reconstruct the reference text while marking the errors

def reconstruct_reference_with_errors(aligned_df):
    # handle None, non-DataFrame, and empty
    if aligned_df is None or not isinstance(aligned_df, pd.DataFrame) or aligned_df.empty:
        return ""
    
    reconstructed = []
    for _, row in aligned_df.iterrows():
        op = row.get('operation')
        ref_word = row.get('reference', '')
        hyp_word = row.get('hypothesis', '')
        if op == '=':
            reconstructed.append(ref_word)
        elif op == 'ins':
            # show the inserted hypothesis word instead of '_' placeholder
            reconstructed.append(f"[INS:{hyp_word}]")
        elif op == 'del':
            reconstructed.append(f"[DEL:{ref_word}]")
        elif op == 'sub':
            reconstructed.append(f"[SUB:{ref_word}->{hyp_word}]")
    return ' '.join(w for w in reconstructed if isinstance(w, str))


def reconstruct_reference_with_medical_errors(aligned_df, ref_medical_indices, hyp_medical_indices):
    """Reconstruct the reference, tagging an error only when it touches a medical
    entity on the human (ref) side OR the ASR (hyp) side.
    - del: tagged when the reference word is medical (ref-side only).
    - sub: tagged when the reference word OR the substituted ASR word is medical (union).
    - ins: tagged only when the inserted ASR word is medical; non-medical insertions
      are omitted entirely (no reference word exists for them).
    Non-medical del/sub are emitted as the plain reference word."""
    if aligned_df is None or not isinstance(aligned_df, pd.DataFrame) or aligned_df.empty:
        return ""

    if ref_medical_indices is None:
        ref_medical_indices = set()
    if hyp_medical_indices is None:
        hyp_medical_indices = set()

    reconstructed = []
    for _, row in aligned_df.iterrows():
        op = row.get('operation')
        ref_word = row.get('reference', '')
        hyp_word = row.get('hypothesis', '')
        ref_ix = row.get('ref_ix')
        hyp_ix = row.get('hyp_ix')
        ref_is_medical = pd.notna(ref_ix) and int(ref_ix) in ref_medical_indices
        hyp_is_medical = pd.notna(hyp_ix) and int(hyp_ix) in hyp_medical_indices

        if op == '=':
            reconstructed.append(ref_word)
        elif op == 'ins':
            if hyp_is_medical:
                reconstructed.append(f"[INS:{hyp_word}]")
        elif op == 'del':
            reconstructed.append(f"[DEL:{ref_word}]" if ref_is_medical else ref_word)
        elif op == 'sub':
            reconstructed.append(
                f"[SUB:{ref_word}->{hyp_word}]" if (ref_is_medical or hyp_is_medical) else ref_word
            )
    return ' '.join(w for w in reconstructed if isinstance(w, str))

# apply to all aligned dfs
all_datasets['whisper_reconstructed_medical_errors'] = all_datasets['whisper_aligned_df'].apply(reconstruct_reference_with_errors)
all_datasets['phi4_reconstructed_medical_errors'] = all_datasets['phi4_aligned_df'].apply(reconstruct_reference_with_errors)
all_datasets['parakeet_reconstructed_medical_errors'] = all_datasets['parakeet_aligned_df'].apply(reconstruct_reference_with_errors)
# all_datasets['granite_reconstructed_ref'] = all_datasets['granite_aligned_df'].apply(reconstruct_reference_with_errors)



In [22]:
def extract_medical_entities_from_ner_tags(ner_tagged_text):
    """Extract medical entity spans from inline NER tags."""
    if pd.isna(ner_tagged_text) or ner_tagged_text is None:
        return []

    ner_tagged_text = str(ner_tagged_text)
    pattern = r'\[([A-Z_]+):\s*([^\]]+)\]'
    entities = []
    plain_index = 0
    cursor = 0

    for match in re.finditer(pattern, ner_tagged_text):
        entity_type = match.group(1)
        entity_text = match.group(2)
        prefix = ner_tagged_text[cursor:match.start()]
        plain_index += len(prefix)
        entities.append({
            'entity_type': entity_type,
            'entity_text': entity_text,
            'start_offset': plain_index,
            'end_offset': plain_index + len(entity_text),
        })
        plain_index += len(entity_text)
        cursor = match.end()

    return entities


def get_medical_word_indices(ner_tagged_text):
    """Return the set of reference word indices (matching ref.split() order used in
    align_words) that fall within a medical entity span tagged in ner_tagged_text."""
    medical_entities = extract_medical_entities_from_ner_tags(ner_tagged_text)
    if not medical_entities:
        return set()

    # Reconstruct the plain text so char offsets line up with the entity spans
    # computed by extract_medical_entities_from_ner_tags.
    tag_pattern = r'\[([A-Z_]+):\s*([^\]]+)\]'
    plain_text = re.sub(tag_pattern, lambda m: m.group(2), str(ner_tagged_text))

    medical_indices = set()
    for word_index, token in enumerate(re.finditer(r'\S+', plain_text)):
        tok_start, tok_end = token.start(), token.end()
        for entity in medical_entities:
            # overlap test between the word span and the entity span
            if tok_start < entity['end_offset'] and entity['start_offset'] < tok_end:
                medical_indices.add(word_index)
                break

    return medical_indices

In [23]:
# Create filtered reconstructed reference columns that only tag medical-entity errors,
# gated by the union of medical entities on the human side (norm_human_transcript_ner)
# and the ASR side (norm_<model>_asr_ner).
all_datasets['whisper_reconstructed_ref_filtered'] = all_datasets.apply(
    lambda row: reconstruct_reference_with_medical_errors(
        row.get('whisper_aligned_df'),
        get_medical_word_indices(row.get('norm_human_transcript_ner')),
        get_medical_word_indices(row.get('norm_whisper_asr_ner')),
    ),
    axis=1
)

all_datasets['phi4_reconstructed_ref_filtered'] = all_datasets.apply(
    lambda row: reconstruct_reference_with_medical_errors(
        row.get('phi4_aligned_df'),
        get_medical_word_indices(row.get('norm_human_transcript_ner')),
        get_medical_word_indices(row.get('norm_phi4_asr_ner')),
    ),
    axis=1
)

all_datasets['parakeet_reconstructed_ref_filtered'] = all_datasets.apply(
    lambda row: reconstruct_reference_with_medical_errors(
        row.get('parakeet_aligned_df'),
        get_medical_word_indices(row.get('norm_human_transcript_ner')),
        get_medical_word_indices(row.get('norm_parakeet_asr_ner')),
    ),
    axis=1
)

print('✓ Created whisper_reconstructed_ref_filtered, phi4_reconstructed_ref_filtered, and parakeet_reconstructed_ref_filtered')

TypeError: reconstruct_reference_with_medical_errors() missing 1 required positional argument: 'hyp_medical_indices'

In [13]:
# column renaming for clarity, all reconstructed_ref to reconstructed_ref_original and all reconstructed_ref_filtered to reconstructed_ref
all_datasets.rename(columns={
    'whisper_reconstructed_ref': 'whisper_reconstructed_ref_original',
    'phi4_reconstructed_ref': 'phi4_reconstructed_ref_original',
    'parakeet_reconstructed_ref': 'parakeet_reconstructed_ref_original',
    'whisper_reconstructed_ref_filtered': 'whisper_reconstructed_ref',
    'phi4_reconstructed_ref_filtered': 'phi4_reconstructed_ref',
    'parakeet_reconstructed_ref_filtered': 'parakeet_reconstructed_ref',
}, inplace=True)

In [9]:
# # Quick validation on first 2 rows before full batch run
# sample_df = all_datasets.head(2).copy()
# sample_df["norm_human_transcript_ner"] = sample_df["norm_human_transcript"].apply(
#     lambda text: apply_ner_to_row(text, PROJECT_ID, LOCATION)
# )

# for i, row in sample_df.iterrows():
#     tagged_text = row["norm_human_transcript_ner"]
#     has_tag = "[UMLS/" in str(tagged_text)
#     print(f"Row {i} has inline UMLS tags: {has_tag}")
#     print(str(tagged_text)[:220])
#     print("-" * 80)

In [14]:
all_datasets.to_excel('results/all_result_processed_normalized_with_ner_tagged.xlsx', index=False, engine='openpyxl')

# load the excel file and display the first few rows
df = pd.read_excel('results/all_result_processed_normalized_with_ner_tagged.xlsx', engine='openpyxl')

# move each models' results to separate sheets in the excel file
with pd.ExcelWriter('results/all_result_separate_sheets_normalized.xlsx', engine='openpyxl') as writer:
    whisper_cols = ['utterance_id', 'source', 'duration_sec', 'human-transcript', 'Whisper-ASR', 'norm_human_transcript', 'norm_human_transcript_ner', 'norm_whisper_asr', 'norm_whisper_asr_ner', 
                    'norm_whisper_asr_wer', 'norm_whisper_asr_ins', 'norm_whisper_asr_del', 
                    'norm_whisper_asr_sub', 'whisper_aligned_df',
                    'norm_whisper_asr_Deletions', 'norm_whisper_asr_Insertions', 'norm_whisper_asr_Substitutions', 'whisper_reconstructed_ref', 'whisper_reconstructed_ref_filtered']
    phi4_cols = ['utterance_id', 'source', 'duration_sec', 'human-transcript', 'Phi-4-ASR', 'norm_human_transcript', 'norm_human_transcript_ner', 'norm_phi4_asr', 'norm_phi4_asr_ner', 
                 'norm_phi4_asr_wer', 'norm_phi4_asr_ins', 'norm_phi4_asr_del', 
                 'norm_phi4_asr_sub', 'phi4_aligned_df',
                 'norm_phi4_asr_Deletions', 'norm_phi4_asr_Insertions', 'norm_phi4_asr_Substitutions', 'phi4_reconstructed_ref', 'phi4_reconstructed_ref_filtered']
    parakeet_cols = ['utterance_id', 'source', 'duration_sec', 'human-transcript', 'Nvidia-Parakeet-ASR', 'norm_human_transcript', 'norm_human_transcript_ner',  'norm_parakeet_asr', 
                     'norm_parakeet_wer', 'norm_parakeet_ins', 'norm_parakeet_del', 
                     'norm_parakeet_sub', 'parakeet_aligned_df',
                     'norm_parakeet_Deletions', 'norm_parakeet_Insertions', 'norm_parakeet_Substitutions', 'parakeet_reconstructed_ref', 'parakeet_reconstructed_ref_filtered']
    # granite_cols = ['utterance_id', 'source', 'duration_sec', 'human-transcript', 'IBM-Granite', 'norm_human_transcript', 'norm_granite', 
    #                 'norm_granite_wer', 'norm_granite_ins', 'norm_granite_del', 
    #                 'norm_granite_sub', 'granite_aligned_df',
    #                 'norm_granite_Deletions', 'norm_granite_Insertions', 'norm_granite_Substitutions', 'granite_reconstructed_ref']
    df_whisper = df.filter(items=whisper_cols, axis=1)
    df_phi4 = df.filter(items=phi4_cols, axis=1)
    df_parakeet = df.filter(items=parakeet_cols, axis=1)
    # df_granite = df.filter(items=granite_cols, axis=1)

    # Only write non-empty DataFrames to avoid invisible sheet error
    if not df_whisper.empty:
        df_whisper.to_excel(writer, sheet_name='Whisper-ASR Results', index=False)
    if not df_phi4.empty:
        df_phi4.to_excel(writer, sheet_name='Phi-4-ASR Results', index=False)
    if not df_parakeet.empty:
        df_parakeet.to_excel(writer, sheet_name='Nvidia-Parakeet-ASR Results', index=False)
    # if not df_granite.empty:
    #     df_granite.to_excel(writer, sheet_name='IBM-Granite Results', index=False)
        
    # resize each row in the sheets to 120px
    for sheet_name in ['Whisper-ASR Results', 'Phi-4-ASR Results', 'Nvidia-Parakeet-ASR Results']: #'Nvidia-Parakeet-ASR Results', 'IBM-Granite Results'
        worksheet = writer.sheets.get(sheet_name)
        if worksheet:
            for row_idx in range(1, len(df) + 2):  # +2 to account for header row and 1-based indexing
                worksheet.row_dimensions[row_idx].height = 120


In [16]:
# select three session with the following utterance ids: day4_consultation07, 1_Malaria, RES0073
selected_utterances = ['2_Diarrhea', '18_Pneumonia', '46aacf84-fdd1-490b-a857-633d2e7763a0_7d4de4c9d3488a4bbd35634cbd3a2b66_l1RjPEwA']
selected_data = all_datasets[all_datasets['utterance_id'].isin(selected_utterances)]

# make all columns lowercase for better readability
selected_data.columns = [col.lower() for col in selected_data.columns]

# rename "phi-4-asr" column to "phi4-asr" for better readability
selected_data.rename(columns={'phi-4-asr': 'phi4-asr'}, inplace=True)

# save the selected data to a new excel file
selected_data.to_excel('results/selected_sessions_normalized_with_ner_tagged.xlsx', index=False, engine='openpyxl')

/tmp/ipykernel_1666650/3316763195.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_data.rename(columns={'phi-4-asr': 'phi4-asr'}, inplace=True)


In [ ]:
# # Example: Filter for specific EntityMention types
# # Available mention types include:
# # ANATOMICAL_STRUCTURE, ASSISTED_LIVING, BF_RESULT, BM_RESULT, BM_UNIT, BM_VALUE,
# # BODY_FUNCTION, BODY_MEASUREMENT, COMPLIANT, DOESNOT_FOLLOWUP, FAMILY, FOLLOWSUP,
# # LABORATORY_DATA, LAB_RESULT, LAB_UNIT, LAB_VALUE, MEDICAL_DEVICE, etc.

# # Define which mention types to include
# MENTION_TYPES = ['ANATOMICAL_STRUCTURE', 'LAB_VALUE', 'LAB_RESULT', 'BM_VALUE']

# # Create a filtered NER column
# all_datasets['norm_human_transcript_ner_filtered'] = all_datasets['norm_human_transcript'].apply(
#     lambda text: apply_ner_to_row(text, PROJECT_ID, LOCATION, mention_types=MENTION_TYPES)
# )

# print(f"✓ Filtered NER tagging complete (types: {MENTION_TYPES})")
# print("\n" + "=" * 80)
# print("Sample output with filtered mention types:")
# print("=" * 80)

# for idx in range(min(2, len(all_datasets))):
#     print(f"\nRow {idx}:")
#     tagged_text = all_datasets['norm_human_transcript_ner_filtered'].iloc[idx]
#     print(f"Filtered: {str(tagged_text)[:250]}...")

✓ Filtered NER tagging complete (types: ['ANATOMICAL_STRUCTURE', 'LAB_VALUE', 'LAB_RESULT', 'BM_VALUE'])

Sample output with filtered mention types:

Row 0:
Filtered: good morning i am doctor smith from babylon can you just confirm your name date of birth and the first line of your address please hi my name is susan  thirty redbridge street sw two two hz hello and your date of birth forty oh two nineteen seventy f...

Row 1:
Filtered:  hello hi i am doctor jacob and welcome to babylon hi  hi so just before we start is it alright if you could confirm your name for me please yep  john doe okay and your date of birth  uhh  twentyone twelve and nineteen  eightysix and your address for...
